# Advanced Model Setup: BERT

## Objective:
Install the Hugging Face `transformers` library and load a pre-trained BERT model for sentiment classification.

In [3]:
# Step 1: Import libraries

from transformers import BertTokenizer, BertForSequenceClassification
import torch

In [4]:
# Step 2: Load pre-trained BERT model and tokenizers

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

print(" BERT model and tokenizer loaded successfully!")

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


 BERT model and tokenizer loaded successfully!


In [5]:
# Step 3: Test the model with a sample test

text = "The movie was absolutely amazing, I loved it!"
inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)

with torch.no_grad():
    outputs = model(**inputs)

print(outputs.logits)

tensor([[-0.2382, -0.7888]])


# Fine-Tuning Bert on IMDB movie Dataset

## Objective:
Fine-tune the pre-traoned BERT model for sentiment classification using the IMDb dataset.

In [6]:
# STep 1: Import libraries

import os
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from tqdm.notebook import tqdm
from tqdm import tqdm
from sklearn.metrics import accuracy_score,precision_score, recall_score, f1_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

In [7]:
# Step 2: Load cleaned IMDb dataset

# Get the absolute path to the data directory
current_dir = os.getcwd()
parent_dir = os.path.dirname(current_dir)
data_path = os.path.join(parent_dir, "data", "cleaned_imdb_reviews.csv")

df = pd.read_csv(data_path)

# Small subset to keep training fast 
df = df.sample(2000, random_state=42).reset_index(drop=True)

df.head()

,review,label,cleaned_review
0,The Little Mermaid is one of my absolute favor...,0,little mermaid one absolute favorite disney mo...
1,saw this in preview- great movie- wonderful ch...,1,saw preview great movie wonderful characteriza...
2,A film with very little positive to say for it...,0,film little positive say itbr br firstly zero ...
3,This made-for-TV film is a brilliant one. This...,1,madefortv film brilliant one probably best fav...
4,Was this meant to be a comedy or a serious dra...,0,meant comedy serious drama film start lighthea...


In [10]:
# Step 3: Tokenization

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

# Tokenize each review
encoding = tokenizer(
    list(df["cleaned_review"]),
    truncation=True,
    padding=True,
    max_length=128,
    return_tensors="pt"
)


In [4]:
# Step 4: Dataset class Preparation

class IMDbDataset(Dataset):
    def __init__(self, encoding, labels):
        self.encodings = encoding
        self.labels = labels
    
    def __getitem__(self, index):
        item = {key: val[index] for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[index])
        return item
    
    def __len__(self):
        return len(self.labels)

In [5]:
# Step 5: Train-test split + Dataloaders

# Split the raw text and labels first
X_train_texts, X_test_texts, y_train, y_test = train_test_split(
    df["cleaned_review"].tolist(), df["label"].values, 
    test_size=0.2, random_state=42
    )

# Tokenize training and test text separately
train_encodings = tokenizer(
    X_train_texts,
    truncation=True,
    padding=True,
    max_length=128,
    return_tensors="pt"
)

test_encodings = tokenizer(
    X_test_texts,
    truncation=True,
    padding=True,
    max_length=128,
    return_tensors="pt"
)

# Create Dataset objects
train_dataset = IMDbDataset(train_encodings, y_train)
test_dataset = IMDbDataset(test_encodings, y_test)

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)



In [6]:
# Step 6 : Load pre-trained model for fine-tuning

model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2 )
optimizer = AdamW(model.parameters(), lr=2e-5)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
# Step 7: Fine-tune BERT

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

model.train()
epochs = 2  # you can increase this later if you want

for epoch in range(epochs):
    print(f"Epoch {epoch+1}/{epochs}")
    total_loss = 0

    # tqdm adds a nice progress bar
    progress_bar = tqdm(train_loader, desc="Training", leave=True)
    
    for batch in progress_bar:
        optimizer.zero_grad()

        # Move tensors to device (GPU or CPU)
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        # Forward pass
        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss

        # Backward pass
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        progress_bar.set_postfix({"Loss": f"{loss.item():.4f}"})

    avg_loss = total_loss / len(train_loader)
    print(f"Average training loss: {avg_loss:.4f}\n")

Epoch 1/2


Training:   8%|▊         | 15/200 [02:08<24:26,  7.92s/it, Loss=0.7400]

In [ ]:
# Step 8: Evaluation

model.eval()  # set model to evaluation mode
correct, total = 0, 0
all_preds, all_labels = [], []

with torch.no_grad():
    progress_bar = tqdm(test_loader, desc="Evaluating", leave=True)
    
    for batch in progress_bar:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids, attention_mask=attention_mask)
        predictions = torch.argmax(outputs.logits, dim=-1)  # <-- correct variable name
        
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

        # Save predictions & labels for metrics
        all_preds.extend(predictions.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

accuracy = correct / total
print(f"Test Accuracy: {accuracy:.4f}")

Evaluating: 100%|██████████| 50/50 [01:02<00:00,  1.26s/it]

Test Accuracy: 0.8150


In [ ]:
# Step 9: Compute metrics
precision = precision_score(all_labels, all_preds)
recall = recall_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds)

print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")

# Optional: full classification report
print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=["Negative", "Positive"]))

# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)

# Plot confusion matrix
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Negative", "Positive"], yticklabels=["Negative", "Positive"])
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()